# step02b_suna_response_correction - does a response-time correction recover the profiles?

This stage TESTS whether a first-order response-time (deconvolution) correction can repair the
lagged SUNA profiles. It estimates a cell time constant tau by the amount of correction that makes
the downcast and upcast overlap, with no bottles used, then checks the result.

Outcome on P45-07: it cannot. No single tau reduces the down/up hysteresis; the mismatch only grows
with tau. That means the lag is not a recoverable first-order response but near-total cell memory
(the gap barely exchanged). Conclusion: use the discrete bottles for nitrate structure, and keep the
homogeneous-only SUNA calibration (offset -0.39, gain 0.89). This notebook is kept as the evidence
for that decision.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
try: get_ipython().run_line_magic("matplotlib", "inline")
except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image=None

try: _anchor = Path(__file__).resolve()
except NameError: _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists(): sys.path.insert(0, str(_p)); break
from project_paths import project_root, product_dir

CRUISE_ID  = "P45_07"
CTD_ROOT   = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
L2_FILE    = product_dir(CRUISE_ID, "suna_l2") / f"{CRUISE_ID}__suna_L2_colocated.csv"
BOTTLE_NO3 = CRUISE_DIR / "metadata" / "bottle_nitrate" / f"{CRUISE_ID}_bottle_nitrate.csv"
OUT_DIR    = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)
L3_DIR     = product_dir(CRUISE_ID, "suna_calibrated");     L3_DIR.mkdir(parents=True, exist_ok=True)

TAU_GRID   = np.arange(2, 82, 2)   # s, candidate time constants
PRESMOOTH  = 5                     # samples, light median pre-smooth before deconvolution
BIN_DB     = 2.0
MIN_FRAMES = 20
DEPTH_WIN  = 2.5
MIN_IN_WIN = 3
# adopted calibration (from step03_suna_bottle_calibration, homogeneous-only fit)
ADOPTED_OFFSET, ADOPTED_GAIN = -0.39, 0.89

L2 = pd.read_csv(L2_FILE, parse_dates=["time_utc"])
bot = pd.read_csv(BOTTLE_NO3)
bot.columns = [c.strip() for c in bot.columns]
if "flag" not in bot.columns: bot["flag"] = ""
bot["flag"] = bot["flag"].fillna("")

use = L2[(L2["in_cast"]==True) & (L2["L1_clean"]==True) & L2["pressure_db"].notna()].copy()
CASTS = []
for cid, g in use.groupby("cast"):
    g = g.sort_values("time_utc")
    if len(g) < MIN_FRAMES: continue
    CASTS.append(dict(cast=cid, t=g["time_utc"].to_numpy(), M=g["nitrate_uM"].to_numpy(),
        p=g["pressure_db"].to_numpy(), leg=g["leg"].to_numpy()))
print(f"{len(CASTS)} casts with >= {MIN_FRAMES} clean in-cast frames")

In [ ]:
def _secs(t): return pd.to_datetime(t).astype("int64").to_numpy() / 1e9

def response_correct(t, M, tau, presmooth=PRESMOOTH):
    """First-order inverse filter along the time series. Returns corrected N."""
    M = np.asarray(M, float)
    if len(M) < 3 or tau <= 0: return M.copy()
    Ms = pd.Series(M).rolling(presmooth, center=True, min_periods=1).median().to_numpy() if presmooth and presmooth>1 else M
    dt = np.diff(_secs(t)); dt = np.clip(dt, 0.1, 10.0)
    a = np.exp(-dt / tau)
    N = Ms.copy()
    N[1:] = (Ms[1:] - a * Ms[:-1]) / (1 - a)
    return N

def leg_binned(p, v, step=BIN_DB):
    d = pd.DataFrame({"p": p, "v": v}).replace([np.inf,-np.inf], np.nan).dropna()
    if d.empty: return pd.Series(dtype=float)
    d["bin"] = (d["p"] // step).astype(int)
    return d.groupby("bin")["v"].median()

def hysteresis_cost(tau):
    cost = 0.0; nb = 0
    for cd in CASTS:
        N = response_correct(cd["t"], cd["M"], tau)
        dn = cd["leg"]=="down"; up = cd["leg"]=="up"
        gd = leg_binned(cd["p"][dn], N[dn]); gu = leg_binned(cd["p"][up], N[up])
        common = gd.index.intersection(gu.index)
        if len(common) >= 3:
            diff = (gd.loc[common] - gu.loc[common]).to_numpy()
            cost += float(np.sum(diff**2)); nb += len(common)
    return cost/nb if nb else np.nan

In [ ]:
raw_rms = np.sqrt(hysteresis_cost(0.0))
rms = np.sqrt(np.array([hysteresis_cost(t) for t in TAU_GRID]))
k = int(np.nanargmin(rms)); TAU_OPT = float(TAU_GRID[k]); best = rms[k]
at_edge = k in (0, len(TAU_GRID)-1)
CORRECTION_WORKS = (best < raw_rms - 0.1) and not at_edge   # a real interior improvement

print(f"raw down/up mismatch (tau=0):  RMS = {raw_rms:.2f} uM")
print(f"best at tau = {TAU_OPT:.0f} s:        RMS = {best:.2f} uM   ({best-raw_rms:+.2f} uM vs raw)")
if CORRECTION_WORKS:
    print(f"\nVERDICT: a response-time correction helps (tau ~ {TAU_OPT:.0f} s). Proceed to the refit in cell 5.")
else:
    why = "the optimum sits at the grid edge" if at_edge else "no tau reduced the mismatch"
    print(f"\nVERDICT: deconvolution does NOT recover the profiles ({why}).")
    print("  The hysteresis is not reduced at any tau, so this is not a recoverable first-order lag")
    print("  but near-total cell memory. Use the discrete bottles for nitrate structure and keep the")
    print(f"  homogeneous-only calibration (offset {ADOPTED_OFFSET}, gain {ADOPTED_GAIN}).")

print("\n tau(s)   RMS_mismatch(uM)")
for t, r in zip(TAU_GRID, rms):
    print(f"  {t:5.0f}     {r:6.2f}{'  <--' if t==TAU_OPT else ''}")

fig = plt.figure(figsize=(6.2,4.2)); ax = fig.add_subplot(111)
ax.plot(TAU_GRID, rms, "-o", ms=3, color="#2E6FB0")
ax.axhline(raw_rms, color="#aaa", ls="--", lw=1, label=f"raw ({raw_rms:.2f})")
ax.axvline(TAU_OPT, color="#d62728", lw=1, label=f"best tau = {TAU_OPT:.0f} s")
ax.set_xlabel("tau (s)"); ax.set_ylabel("down/up RMS mismatch (uM)")
ax.set_title(f"{CRUISE_ID} SUNA response-time test"); ax.legend(fontsize=8, frameon=False)
for s in ("top","right"): ax.spines[s].set_visible(False)
fig.tight_layout(); out = OUT_DIR/f"{CRUISE_ID}__suna_tau_sweep.png"; fig.savefig(out, dpi=135)
try: plt.close(fig)
except Exception: pass
if Image is not None:
    try: display(Image(filename=str(out)))
    except Exception: pass
print("saved", out.name)

In [ ]:
SHOW = ["P45_07_CTD_10", "P45_07_CTD_16", "P45_07_CTD_09"]
cast_by = {c["cast"]: c for c in CASTS}
for cid in SHOW:
    if cid not in cast_by: continue
    cd = cast_by[cid]; N = response_correct(cd["t"], cd["M"], TAU_OPT)
    dn = cd["leg"]=="down"; up = cd["leg"]=="up"
    bd = bot[(bot["cast_id"]==cid) & (bot["flag"].astype(str).str.strip()=="")]
    fig = plt.figure(figsize=(5.2,6.4)); ax = fig.add_subplot(111)
    ax.plot(cd["M"][dn], cd["p"][dn], ".", ms=2, color="#bcd3ea", label="raw down")
    ax.plot(cd["M"][up], cd["p"][up], ".", ms=2, color="#f3cbb0", label="raw up")
    ax.plot(N[dn],       cd["p"][dn], ".", ms=2.6, color="#2E6FB0", label="corr down")
    ax.plot(N[up],       cd["p"][up], ".", ms=2.6, color="#E07B39", label="corr up")
    if len(bd): ax.plot(bd["nitrate_uM"], bd["depth_m"], "s", ms=8, color="#d62728", label="bottle", zorder=5)
    ax.invert_yaxis(); ax.axvline(0, color="#bbb", lw=0.5)
    ax.set_xlabel("NO3 (uM, raw SUNA scale)"); ax.set_ylabel("pressure (db)")
    ax.set_title(f"{cid}  tau={TAU_OPT:.0f}s", fontsize=10); ax.legend(fontsize=7, frameon=False)
    for s in ("top","right"): ax.spines[s].set_visible(False)
    fig.tight_layout(); out = OUT_DIR/f"{CRUISE_ID}__suna_respcorr__{cid}.png"; fig.savefig(out, dpi=130)
    try: plt.close(fig)
    except Exception: pass
    if Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: pass
    print("saved", out.name)

In [ ]:
cast_by = {c["cast"]: c for c in CASTS}
if not CORRECTION_WORKS:
    print("Response correction did not work (see the cell 3 verdict), so no recalibration is done.")
    print("Adopted SUNA calibration stays the homogeneous-only fit from step03:")
    print(f"  SUNA = {ADOPTED_OFFSET} + {ADOPTED_GAIN} * bottle   (R2 0.997, confirmed by P45-06's 0.884)")
else:
    corr = {cd["cast"]: response_correct(cd["t"], cd["M"], TAU_OPT) for cd in CASTS}
    def suna_corr_at(cast, depth, win=DEPTH_WIN):
        if cast not in cast_by: return (np.nan, 0)
        cd = cast_by[cast]; N = corr[cast]
        m = (cd["p"] >= depth-win) & (cd["p"] <= depth+win)
        return (float(np.median(N[m])), int(m.sum())) if m.sum() else (np.nan, 0)
    rows=[]
    for _, r in bot.iterrows():
        if str(r["flag"]).strip() != "": continue
        s, n = suna_corr_at(r["cast_id"], float(r["depth_m"]))
        rows.append(dict(cast=r["cast_id"], depth_m=r["depth_m"], bottle=r["nitrate_uM"],
                         suna_corr=round(s,3) if np.isfinite(s) else np.nan, n=n,
                         used=(n>=MIN_IN_WIN and np.isfinite(s))))
    cc = pd.DataFrame(rows); fitc = cc[cc["used"]]
    print(f"{len(fitc)} usable bottle pairs after response correction\n")
    if len(fitc) >= 2:
        b = fitc["bottle"].to_numpy(); s = fitc["suna_corr"].to_numpy()
        (g, o), cov = np.polyfit(b, s, 1, cov=True)
        pred = o + g*b; r2 = 1 - np.sum((s-pred)**2)/np.sum((s-s.mean())**2)
        rmse = float(np.sqrt(np.mean(((s-o)/g - b)**2)))
        print(f"response-corrected calibration:  SUNA = {o:.3f} + {g:.4f} * bottle | R2 {r2:.4f} | RMSE {rmse:.3f} uM | n {len(fitc)}")
    cc.to_csv(OUT_DIR/f"{CRUISE_ID}__suna_respcorr_calibration.csv", index=False, encoding="utf-8-sig")